## 0. Project setup
Create a repo with folders data/, notebooks/, src/, configs/, results/.
Make an environment with torch, torchaudio, transformers, datasets, librosa, scikit-learn, matplotlib, optuna.
Fix random seeds, and log package versions and every run’s settings (a config.yaml per run, plus Weights & Biases or MLflow if you like).
Start a manifest.csv that records every file’s source, label and split.

## 1. Data collection
Load DADS: datasets.load_dataset("geronimobasso/drone-audio-detection-samples").
Record each clip’s origin where you can (the dataset card lists the source datasets).
Add distractor data (ESC-50, UrbanSound8K, FSD50K subsets) and, later, your own recordings as a final test set.
Note each source’s license in the manifest.

## 2.1 Exploratory data analysis
Count clips per class and per source, and plot clip-length histograms (lengths run from 0.02 s to 228 s).
Check sample rates and channels, and flag silent, clipped or corrupt files.
Plot average spectrograms for drone and non-drone, and listen to about 30 random clips per class.
Check for duplicates (audio hashes) and for shortcuts: can a simple model guess the source from the audio alone? If so, fix that before modeling.

## 2.2 Data pre-processing
Convert everything to mono, 16 kHz.
Cut into fixed windows (1 s, 0.5 s hop) and drop windows that are silent or too short. Keep the source ID on every window.
Remove microphone-on-drone sources (DREGON, SPCup19) if your detector is ground-based.
Split by source or recording, never by clip: about 70% train, 15% validation, 15% test, with one source held out entirely as an extra test.
Balance classes with weights or a balanced sampler, not by duplicating clips.
Training-only augmentation: random gain, time shift, mixing drone clips onto backgrounds at SNRs from about -10 to +20 dB, and SpecAugment.

## 3. Feature extraction
Convert each window to a log-mel spectrogram (128 mel bins, 25 ms window, 10 ms hop). The Hugging Face ASTFeatureExtractor does this.
Normalize with the dataset mean and standard deviation.
AST expects 10-second inputs by default. For 1-second windows, set a shorter max_length (about 100 frames), or pad or stack windows to 2-3 s. When you change the length, load the model with ignore_mismatched_sizes=True so its position embeddings are resized.
Cache the spectrograms or compute them on the fly in a DataLoader.

## 4. Transformer model
Start from a pretrained AST (MIT/ast-finetuned-audioset-10-10-0.4593) and replace the head with a 2-class output (drone, not drone).
Compare against a small baseline (a CNN or the MFCC random forest) so the transformer’s gain is measured.
Alternatives if AST is too heavy: PANNs CNN14, or a small audio transformer for edge devices.

## 5. Training
Loss: cross-entropy with class weights (or focal loss if drones are rare).
Optimizer: AdamW, learning rate about 1e-5 to 5e-5 for the transformer, with warmup and cosine decay.
First train only the head for a few epochs, then unfreeze the whole model.
Batch size 16-64 with mixed precision. Early-stop on validation ROC-AUC or PR-AUC.
Save the best checkpoint and the training curves (loss and accuracy per epoch).

## 6. Evaluation (baseline transformer)
On the held-out test set, report accuracy, precision, recall, F1, ROC-AUC and PR-AUC.
Plot the ROC curve, precision-recall curve, confusion matrix and the training curves.
Report detection rate at a fixed false-alarm rate (for example 1% and 5%) and pick a threshold, since false alarms matter for operators.
Break results down by source and by SNR. Look at the examples the model got wrong.
Evaluate on the held-out source and on your own recordings to see the real-world gap.

## 7. Optimization
Tune with Optuna: learning rate, batch size, window length, augmentation strength, number of frozen layers, dropout and weight decay.
Try layer-wise learning-rate decay and stronger noise mixing for hard conditions.
Calibrate the output (temperature scaling) and retune the threshold on validation data.
Compress for deployment: distillation to a smaller model, quantization, ONNX export. Measure latency and size.
Change one thing at a time, and keep a results table of every run.

## 8. Evaluation (optimized model)
Repeat all of step 6 on the same test sets, with the same plots.
Show baseline versus optimized side by side: ROC-AUC, PR-AUC, recall at fixed false alarms, latency and model size.
Report confidence intervals (bootstrap over recordings) and note whether the gain is bigger than the noise.
Write down limits: sources not covered, microphones tested, distances, and the false-alarm types you saw.